# Exploration de l'API TED (marchés publics européens)

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)
**Module :** 1 - Collecte
**Source :** TED (Tenders Electronic Daily), le journal officiel des marchés publics de l'UE.

## Pourquoi TED

La BCEAO publie très peu de cybersécurité. Je cherche une source avec du volume et de vraies
opportunités cyber. TED est une **API officielle** : données structurées (pas de HTML à scraper),
énorme volume (toute l'Europe), filtrage par mots-clés / pays / catégorie.

## Ma démarche

Je ne connais pas cette API. Je la découvre **par tâtonnements** : je tente un appel, je lis ce que
l'API répond (surtout ses erreurs, très instructives), et j'ajuste. Je veux qu'aucun nom de champ
n'apparaisse "par magie" : chaque nom que j'utilise, je montre d'abord d'où il vient.

## 1. Est-ce que l'API répond ?

Je fais l'appel le plus simple possible et je regarde le **code HTTP** (200 = OK, 4xx = j'ai fait
une erreur, 5xx = souci serveur).

In [ ]:
import requests
import json
from collections import Counter
from datetime import date

URL = "https://api.ted.europa.eu/v3/notices/search"

r = requests.post(URL, json={"query": "publication-date>=today(-1)"})
print("Code HTTP :", r.status_code)

**400** : l'API n'est pas contente. Ce n'est pas un échec, c'est une piste : elle va me dire
ce qui manque. Je lis le message.

In [ ]:
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

Le message dit : le champ **`fields` ne doit pas être vide**. L'API veut que je précise quelles
informations je veux pour chaque annonce. Mais je ne connais aucun nom de champ valide... Je vais
en tenter un au hasard pour voir comment l'API réagit.

## 2. Découvrir les noms de champs (grâce à l'erreur)

Je tente un nom intuitif au hasard : `"title"`.

In [ ]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)", "fields": ["title"]})
print("Code HTTP :", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False)[:350])

Encore 400, mais le message est en or : « unsupported value (**supported values are:** ... ) »
suivi d'une longue liste. En me trompant, l'API m'a **listé tous les champs valides**. Je récupère
cette liste et je la compte.

In [ ]:
message = r.json()["message"]
liste_brute = message.split("supported values are:")[1].rstrip(")").strip()
champs = [c.strip() for c in liste_brute.split(",") if c.strip()]
print("Nombre de champs proposés par l'API :", len(champs))
print("Exemples (10 premiers) :", champs[:5], "...")

**Attention :** ces 1830, ce sont des **champs** (les colonnes possibles : titre, date, pays...),
PAS le nombre d'annonces. Je verrai le nombre d'annonces plus loin. Pour l'instant, je cherche dans
cette liste les champs dont j'ai besoin.

## 3. Chercher mes champs dans la liste (avant de les utiliser)

Je ne veux pas utiliser un nom de champ sans l'avoir vu dans la liste. Donc je fouille la liste avec
des mots-clés correspondant aux informations utiles : identifiant, titre, date, pays, lien, catégorie.
J'écarte les champs préfixés `BT-` (codes techniques peu lisibles) juste pour la lisibilité.

In [ ]:
for mot in ["publication", "number", "title", "deadline", "buyer-country", "buyer-name", "links", "cpv"]:
    trouves = [c for c in champs if mot in c.lower() and not c.startswith("BT-")][:5]
    print(f"{mot:16} -> {trouves}")

Maintenant je vois d'où viennent les noms que je vais utiliser. En particulier,
**`publication-number`** apparaît bien dans la liste (sous "publication" et "number") : c'est
l'identifiant de l'annonce. Je retiens : `publication-number`, `notice-title`, `buyer-name`,
`buyer-country`, `publication-date`, `deadline`, `classification-cpv`, `links`.

## 4. Premier appel qui marche, et surtout : que renvoie l'API ?

J'utilise `publication-number` (trouvé dans la liste) pour faire un appel valide. Puis, au lieu de
supposer la forme de la réponse, je **regarde les clés** que l'API me renvoie.

In [ ]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)",
                              "fields": ["publication-number"], "limit": 3})
reponse = r.json()
print("Code HTTP :", r.status_code)
print("\nLes CLÉS que l'API me renvoie :")
for cle, val in reponse.items():
    apercu = f"liste de {len(val)} éléments" if isinstance(val, list) else val
    print(f"   {cle} = {apercu}")

Voilà la structure de la réponse, sans rien deviner :
- **`notices`** : la liste des annonces demandées (ici 3, à cause de `limit=3`) ;
- **`totalNoticeCount`** : le nombre **total** d'annonces correspondant à ma recherche ;
- **`iterationNextToken`** : un jeton pour récupérer la suite (pagination) ;
- **`timedOut`** : si la requête a expiré.

C'est ici que je découvre **`totalNoticeCount`** : ce n'est pas un champ que je demande, c'est une
clé que l'API met dans sa réponse. C'est lui qui me donne le nombre de **lignes** (annonces).

## 5. Combien d'annonces existent ? (lignes, pas champs)

Je me fais une petite fonction pour lire `totalNoticeCount` selon une recherche.

In [ ]:
def total_annonces(query, scope="ACTIVE"):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": scope}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Annonces publiées depuis 1 jour  :", total_annonces("publication-date>=today(-1)"))
print("Annonces publiées depuis 7 jours :", total_annonces("publication-date>=today(-7)"))

Plusieurs milliers par jour. Sur le **site** TED, sans filtre, j'obtiens ~6 millions. Pourquoi
l'écart ? Parce que le site montre **tout l'historique**, alors que je filtre sur les derniers jours.
Je le vérifie en enlevant le filtre de date (scope ALL).

In [ ]:
def total_all(query):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": "ALL"}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Tout l'historique :", total_all("publication-date>=today(-9000)"))
print("Depuis 1 an       :", total_all("publication-date>=today(-365)"))
print("Depuis 1 jour     :", total_all("publication-date>=today(-1)"))

Confirmé : tout l'historique donne ~6-7 millions, comme le site. L'API et le site sont donc
cohérents. **Conséquence :** pour ma veille, je filtrerai toujours par date récente + cyber + scope
ACTIVE, et je ferai tourner la collecte chaque jour (sinon le volume est ingérable).

## 6. Une fonction de recherche réutilisable

In [ ]:
CHAMPS = ["publication-number", "notice-title", "buyer-name", "buyer-country",
          "publication-date", "deadline", "notice-type", "classification-cpv", "links"]

def chercher(query, limit=50, scope="ACTIVE"):
    body = {"query": query, "fields": CHAMPS, "limit": limit, "page": 1, "scope": scope}
    r = requests.post(URL, json=body, timeout=40)
    r.raise_for_status()
    return r.json()

## 7. Regarder une annonce en détail

Avant de filtrer, je regarde la structure brute d'une annonce (c'est là qu'on trouve les surprises).

In [ ]:
data = chercher("publication-date>=today(-2)", limit=5)
print(json.dumps(data["notices"][0], indent=2, ensure_ascii=False)[:1000])

J'observe trois formats à gérer : le **titre** est un dictionnaire par langue, les **dates** ont
un fuseau (ex. `2026-06-18+02:00`), et `classification-cpv` est une **liste** de codes.

## 8. Isoler la cybersécurité : la nomenclature CPV officielle

Sur le site, à gauche, des filtres regroupent les annonces par grandes catégories (« Computer and
Related Services »...). En cliquant, j'ai vu qu'elles correspondent à des **codes CPV**. Mais le site
affiche aussi des **nombres** d'annonces par catégorie, et je ne veux PAS identifier mes codes à
partir de ces nombres : ils changent chaque jour (de nouvelles annonces arrivent), donc ce ne serait
pas reproductible.

La source **rigoureuse et stable**, c'est la **nomenclature CPV officielle** de l'UE : un fichier qui
associe chaque code à son libellé, et qui ne change pas dans le temps (version 2008, toujours en
vigueur). Je l'ai téléchargé une fois depuis TED/SIMAP et rangé dans mon projet. Je le charge.

In [ ]:
import pandas as pd

# Fichier officiel CPV téléchargé depuis TED/SIMAP (adapte le chemin si besoin)
CHEMIN_CPV = "data/cpv_2008.xlsx"
cpv = pd.read_excel(CHEMIN_CPV)
print("Dimensions (codes, langues) :", cpv.shape)
print("Colonnes (langues) :", list(cpv.columns)[:12], "...")
cpv.head(3)

9454 codes, une colonne par langue (EN, FR, etc.). Je cherche les codes dont le **libellé anglais**
contient des mots liés à mon métier : audit, security, testing, penetration.

In [ ]:
mots = ["audit", "security", "testing", "penetration"]
masque_mot = cpv["EN"].str.lower().str.contains("|".join(mots), na=False)
print("Codes dont le libellé contient ces mots :", masque_mot.sum())
# aperçu rapide
for _, l in cpv[masque_mot].head(8).iterrows():
    print(f"  {str(l['CODE']).split('-')[0]:10} | {l['EN']}")

Attention : il y en a beaucoup (~85), mais avec du **bruit** : « Blood-testing reagents »,
« Convex security mirrors », « Cybercafé services »... Le mot « testing » ou « security » apparaît
dans des domaines qui n'ont rien à voir avec la cybersécurité. Je dois donc **trier**.

La cybersécurité informatique est surtout dans la **branche 72** (services informatiques) et quelques
codes **487** (logiciels de sécurité). Je combine donc : libellé pertinent **ET** code dans ces
branches.

In [ ]:
cpv["code8"] = cpv["CODE"].str.split("-").str[0]
masque_info = cpv["code8"].str.startswith("72") | cpv["code8"].str.startswith("487")
pertinents = cpv[masque_mot & masque_info]

print("Codes CPV retenus pour la cybersécurité :\n")
for _, l in pertinents.iterrows():
    print(f"  {l['code8']:10} | {l['EN']}")

# Je garde la liste des codes pour ma requête de collecte
CODES_CPV_CYBER = pertinents["code8"].tolist()

Voilà mes codes, trouvés **par leur signification** (libellés officiels), sans dépendre d'aucun
nombre. Ils sont stables dans le temps. Les plus pertinents : `72810000` (audit informatique),
`72820000` (essai informatique), `72800000` (audit et essai), `72254000` (essais de logiciels),
`48730000` (logiciels de sécurité), `72212730` (développement de logiciels de sécurité)...

**Limite à garder en tête :** une annonce de pentest peut être classée ailleurs (services
informatiques généraux, conseil...). Ces codes sont donc un **signal fort**, pas l'unique filtre. Je
les combinerai avec des **mots-clés** dans le titre, et le **scoring IA** tranchera.

## 9. Ma requête de collecte cyber (CPV officiels + mots-clés)

Je combine les codes CPV trouvés ci-dessus (signal fort) ET des mots-clés dans le titre, en me
limitant aux mises en concurrence (`notice-type=cn-standard`), c'est-à-dire des appels d'offres
ouverts (pas des avis d'attribution déjà clôturés).

In [ ]:
# Construire la partie CPV de la requête à partir des codes trouvés
filtre_cpv = " OR ".join(f"classification-cpv={c}" for c in CODES_CPV_CYBER)
filtre_mots = ('notice-title ~ "cybersecurity" OR notice-title ~ "penetration" '
               'OR notice-title ~ "ISO 27001" OR notice-title ~ "security audit" '
               'OR notice-title ~ "vulnerability" OR notice-title ~ "pentest"')

REQUETE_CYBER = (f"({filtre_cpv} OR {filtre_mots}) "
                 "AND notice-type=cn-standard SORT BY publication-date DESC")

data = chercher(REQUETE_CYBER, limit=100)
print("Annonces cyber (mises en concurrence) :", data["totalNoticeCount"])
print("Récupérées dans l'échantillon :", len(data["notices"]))

## 10. Contrôle qualité des données

Avant de coder le collecteur, je vérifie la qualité sur cet échantillon réel.

In [ ]:
notices = data["notices"]
n = len(notices)
print(f"Échantillon : {n} annonces\n")
print("a) Complétude :")
for champ in CHAMPS:
    presents = sum(1 for x in notices if x.get(champ) not in (None, "", [], {}))
    print(f"   {champ:20} : {presents}/{n}  ({100*presents//max(n,1)}%)")

In [ ]:
print("b) Doublons sur publication-number :")
nums = [x.get("publication-number") for x in notices]
print(f"   {len(nums)} numéros, {len(set(nums))} uniques -> {len(nums)-len(set(nums))} doublon(s)")

In [ ]:
print("c) Langues des titres :")
sans_anglais = sum(1 for x in notices
                   if isinstance(x.get("notice-title"), dict) and "eng" not in x["notice-title"])
print(f"   annonces sans titre anglais : {sans_anglais}/{n}")

In [ ]:
print("d) Date limite (deadline) :")
def premiere(d): return d[0] if isinstance(d, list) else d
avec_dl = [x for x in notices if x.get("deadline")]
print(f"   présente dans : {len(avec_dl)}/{n}")
if avec_dl:
    print("   exemple de format :", premiere(avec_dl[0]["deadline"]))
    auj = date.today().isoformat()
    passees = sum(1 for x in avec_dl if str(premiere(x["deadline"]))[:10] < auj)
    print(f"   déjà passées alors que scope=ACTIVE : {passees}/{len(avec_dl)}")

In [ ]:
print("e) Lien : je le reconstruis depuis le numéro")
num = notices[0]["publication-number"]
print("   ", f"https://ted.europa.eu/fr/notice/{num}")

### Ce que j'apprends

- Champs **fiables à 100 %** : `publication-number`, `notice-title`, `buyer-name`,
  `buyer-country`, `publication-date`, `classification-cpv`, `links`.
- **`publication-number` unique** (0 doublon) -> ma clé.
- **Titres toujours en anglais** -> je prends `eng`.
- **`deadline` souvent absente, et parfois déjà passée même en scope ACTIVE.** Je ne m'y fie donc
  pas seule : si elle est présente et dépassée, l'annonce est close ; sinon je me base sur `ACTIVE`.
  Format à normaliser : `2026-06-29T15:00:00+01:00` -> `2026-06-29`.
- **Liens** : structure complexe -> je reconstruis `https://ted.europa.eu/fr/notice/<num>`.

## 11. Les détails riches (prix, procédure, gagnant) sont-ils dans l'API ?

En cliquant sur une annonce du site, j'ai vu plein de détails : valeur estimée (le prix), type de
procédure, durée, gagnant, email de l'acheteur. Est-ce que l'API a ces champs ? Je cherche dans la
liste.

In [ ]:
for label, ms in {"valeur/prix":["estimated-value","value-cur"], "procédure":["procedure-type"],
                  "durée":["contract-duration"], "gagnant":["winner"],
                  "description":["description-lot","description-proc"]}.items():
    trouves = []
    for m in ms:
        trouves += [c for c in champs if m in c.lower() and not c.startswith("BT-")]
    print(f"{label:14} -> {trouves[:4]}")

Oui, l'API contient ces détails. Donc **pas besoin de scraper la page de détail** : tout est
dans l'API, il suffit d'ajouter les champs voulus à ma requête.

**Stratégie (la même que pour la BCEAO) :** je ne récupère pas tout pour tout le monde.
- À la collecte : champs **essentiels** (titre, organisme, pays, date, lien, CPV) pour toutes les
  annonces ouvertes.
- Après le scoring : champs **riches** (prix, procédure, durée, description, documents) seulement
  pour les annonces jugées **pertinentes**, en enrichissant la requête API.

Remarque : un avis avec une section « Results / Winner » est un **avis d'attribution** (marché déjà
gagné), pas une opportunité. Le filtre `notice-type=cn-standard` les écarte.

## 12. Décisions de conception pour `collecter_ted()`

Pour alimenter la même base que la BCEAO, je produis le même format :

| Champ de ma base | Source TED |
|---|---|
| `cle_unique` | `ted::<publication-number>` |
| `intitule` | `notice-title["eng"]` |
| `source` | `"TED"` |
| organisme / pays | `buyer-name` / `buyer-country` |
| `date_publication` | `publication-date` normalisée |
| `delai_soumission` | `deadline` si présent, normalisé, sinon `null` |
| `lien` | `https://ted.europa.eu/fr/notice/<numéro>` |
| `statut_source` | `"en cours"` (ACTIVE), ou `"clôturé"` si date limite dépassée |

Le prix et les détails riches seront ajoutés **plus tard**, pour les annonces pertinentes.

## 13. Prochaines étapes

1. Écrire `collecter_ted()` avec les champs **essentiels**, testée sur de vraies données.
2. L'intégrer dans `collecte_et_scoring.py`, à côté de `collecter_bceao()`.
3. Vérifier que le scoring repère les vraies annonces cyber.
4. Plus tard : ajouter le prix et les détails pour les pertinentes ; gérer la pagination
   (`iterationNextToken`) si besoin de plus de 100 annonces.